# Team B — Mistral 7B: Profiling + Roofline Analysis

**Runtime:** GPU → A100 (Runtime → Change runtime type → A100)  
**Estimated time:** 2-3 hours  
**GPU memory needed:** ~14 GB for FP16, ~4 GB for INT4

## What this notebook does
1. Installs dependencies
2. Benchmarks per-token latency for each precision config
3. Measures GPU memory footprint and KV-cache usage
4. Runs PyTorch Profiler and exports Chrome trace
5. Computes Roofline model (compute vs memory bound)
6. Generates comparison plots
7. Saves results to Google Drive

## 0. Setup

In [ ]:
!nvidia-smi
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pip install -q transformers datasets accelerate tokenizers
!pip install -q auto-gptq autoawq bitsandbytes
!pip install -q matplotlib scipy wandb
print('Done.')

In [ ]:
!git clone https://github.com/YOUR-ORG/uncertainty-aware-inference.git
%cd uncertainty-aware-inference
import sys
sys.path.insert(0, '.')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/uncertainty-inference/profiling'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results → {RESULTS_DIR}')

## 1. Latency Benchmark

In [ ]:
from src.quantization.loaders import load_model, free_model
from src.profiling.harness import benchmark_latency, measure_memory, profile_model
import matplotlib.pyplot as plt
import numpy as np
import json

MODEL_ID   = 'mistralai/Mistral-7B-v0.1'
MODEL_NAME = 'Mistral-7B'
DEVICE     = 'cuda'
PRECISIONS = ['FP16', 'GPTQ_INT4', 'AWQ_INT4', 'NF4']

all_profiling_results = []

for precision in PRECISIONS:
    print(f'\n{"="*50}\n  {precision}\n{"="*50}')
    try:
        model, tokenizer = load_model(MODEL_ID, precision)
        print(f'  GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB')

        result = profile_model(
            model      = model,
            tokenizer  = tokenizer,
            model_name = MODEL_NAME,
            precision  = precision,
            output_dir = RESULTS_DIR,
            device     = DEVICE,
            run_profiler = (precision == 'FP16'),  # only trace FP16 to save time
        )
        all_profiling_results.append(result)
        print(result.summary())

    except Exception as e:
        print(f'  ERROR: {e}')
    finally:
        free_model(model)

## 2. Summary Comparison

In [ ]:
if all_profiling_results:
    labels = [r.precision      for r in all_profiling_results]
    tps    = [r.tokens_per_sec for r in all_profiling_results]
    mem    = [r.gpu_mem_peak_gb for r in all_profiling_results]
    lat    = [r.latency_ms_mean for r in all_profiling_results]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(labels)))

    for ax, (vals, title, ylabel) in zip(axes, [
        (tps,  'Throughput',       'Tokens / Second'),
        (mem,  'Peak GPU Memory',  'GB'),
        (lat,  'Mean Latency',     'ms / token'),
    ]):
        bars = ax.bar(labels, vals, color=colors)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() * 1.02,
                    f'{v:.1f}', ha='center', fontsize=10)
        ax.set_title(title, fontsize=12)
        ax.set_ylabel(ylabel)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle(f'Profiling Summary — {MODEL_NAME}', fontsize=13, fontweight='bold')
    fig.tight_layout()
    fig.savefig(f'{RESULTS_DIR}/{MODEL_NAME}_profiling_summary.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\n=== Profiling Summary ===')
    print(f'{"Precision":12s}  {"tok/s":>8s}  {"lat_ms":>8s}  {"p99_ms":>8s}  {"mem_GB":>8s}  {"Bound":>8s}')
    print('-' * 65)
    for r in all_profiling_results:
        bound = 'MEMORY' if r.is_memory_bound else 'COMPUTE'
        print(f'{r.precision:12s}  {r.tokens_per_sec:8.1f}  '
              f'{r.latency_ms_mean:8.2f}  {r.latency_ms_p99:8.2f}  '
              f'{r.gpu_mem_peak_gb:8.2f}  {bound:>8s}')

## 3. Roofline Plot

In [ ]:
from src.profiling.harness import plot_roofline

if len(all_profiling_results) > 1:
    fig = plot_roofline(
        all_profiling_results,
        save_path=f'{RESULTS_DIR}/{MODEL_NAME}_roofline.png'
    )
    plt.show()
    print('Roofline plot saved.')

    print('\n=== Roofline Analysis ===')
    for r in all_profiling_results:
        bound = 'MEMORY-BOUND' if r.is_memory_bound else 'COMPUTE-BOUND'
        print(f'  {r.precision:12s}: AI={r.arithmetic_intensity:.1f} FLOP/B  '
              f'(ridge={r.ridge_point:.1f})  → {bound}')

## 4. Save Results

In [ ]:
from dataclasses import asdict
summary = {
    'model': MODEL_NAME,
    'results': {r.precision: asdict(r) for r in all_profiling_results}
}
with open(f'{RESULTS_DIR}/{MODEL_NAME}_profiling_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('All profiling results saved to Google Drive.')
print('Upload these JSON files to the shared repo before Team C runs analysis.')